In [1]:
import os
#os.environ["HF_HOME"] = "/data/cat/ws/kafu622g-propaganda/huggingface"
os.environ["HF_HOME"] = "../hf_models"
os.environ["VLLM_TARGET_DEVICE"] = "cpu"

In [2]:
import os
print("VLLM_TARGET_DEVICE:", os.environ.get("VLLM_TARGET_DEVICE"))
import vllm
print("vLLM version:", vllm.__version__)

VLLM_TARGET_DEVICE: cpu
vLLM version: 0.15.1


In [3]:
import json
import re
import pandas as pd

from vllm import LLM, SamplingParams

os.chdir(os.path.expanduser("~/Github/masterthesis/analysis/hpc_code_tweets")) 
data_path = "./"

INFO 02-26 00:21:22 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 02-26 00:21:22 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


In [4]:
bt_follow_reftweets = pd.read_csv("bt_follow_2022-02-07_2022-02-14_reftweets_dedup.csv")
bt_follow_tweets = pd.read_csv("bt_follow_2022-02-07_2022-02-14_tweets.csv")
bt_track_reftweets = pd.read_csv("bt_track_2022-02-07_2022-02-14_reftweets_dedup.csv")
bt_track_tweets = pd.read_csv("bt_track_2022-02-07_2022-02-14_tweets.csv")

d = pd.concat([bt_follow_reftweets, bt_follow_tweets, bt_track_reftweets, bt_track_tweets], ignore_index=True)
d = d.drop_duplicates(subset="id")

In [5]:
# read prompt file
with open("prompt-populism.md", "r", encoding="utf-8") as f:
  prompt_populism = f.read()

# LLM CODING

## Process Posts with LLM

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME1 = "../hf_models/Qwen3-0.6B"
MODEL_NAME2 = "../hf_models/Qwen3-4B-Instruct-2507"
MODEL_NAME3 = "../hf_models/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME1)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME1, 
    torch_dtype=torch.float32,
    attn_implementation="sdpa"
)
model = torch.compile(model)
torch.set_num_threads(torch.get_num_threads())

temp_sample = d.sample(n=10)[["id", "text"]].reset_index(drop=True)

# PROCESS TEXT
prompts_list = []
rows_list = []
for row in temp_sample.itertuples():
    conversation = [
        {"role": "system", "content": prompt_populism},
        {"role": "user", "content": row.text}
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        conversation, 
        tokenize=False, 
        add_generation_prompt=True
    )
    prompts_list.append(formatted_prompt)
    rows_list.append(row)

print(f"Starting inference on {len(prompts_list)} items...")

outputs = []
for i, prompt in enumerate(prompts_list):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=4048, temperature=0.2, do_sample=True)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    outputs.append(response)
    print(f"  Done {i+1}/{len(prompts_list)}")

`torch_dtype` is deprecated! Use `dtype` instead!


Starting inference on 10 items...
  Done 1/10
  Done 2/10
  Done 3/10
  Done 4/10
  Done 5/10
  Done 6/10
  Done 7/10
  Done 8/10
  Done 9/10
  Done 10/10


In [5]:
MODEL_NAME1 = "../hf_models/Qwen3-0.6B"
MODEL_NAME2 = "../hf_models/Qwen3-4B-Instruct-2507"
MODEL_NAME3 = "../hf_models/Qwen3-8B"

llm_kwargs3 = dict(
    model = MODEL_NAME3,
    # CPU SETTINGS
    dtype="float32",
    enforce_eager = True  
)

sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=4048,
    #top_p=0.9
)

llm = LLM(**llm_kwargs3)
tokenizer = llm.get_tokenizer()

temp_sample = d.sample(n=10)[["id", "text"]].reset_index(drop=True)

# PROCESS TEXT
prompts_list = []
rows_list = []

for row in temp_sample.itertuples():

    conversation = [
        {"role": "system", "content": prompt_populism},
        {"role": "user", "content": row.text}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        conversation, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    prompts_list.append(formatted_prompt)
    rows_list.append(row)


print(f"Starting inference on {len(prompts_list)} items...")
outputs = llm.generate(prompts_list, sampling_params)

INFO 02-25 23:33:48 [utils.py:261] non-default args: {'dtype': 'float32', 'disable_log_stats': True, 'enforce_eager': True, 'model': '../hf_models/Qwen3-8B'}


RuntimeError: Device string must not be empty

## Process Results

In [ ]:
d_results = []

for row, output in zip(rows_list, outputs):
    answer = output.outputs[0].text
    
    # --- 1. Attempt Standard JSON Parsing ---
    try:
        # Isolate the JSON object (find first { and last })
        json_str = answer[answer.find("{"):answer.rfind("}") + 1]
        answer_data = json.loads(json_str)
        
    except (json.JSONDecodeError, ValueError, AttributeError):
        # --- 2. Regex Fallback (Fixes "Bad Quotes" inside text) ---
        # The model often breaks JSON by putting double quotes inside strings.
        # We manually extract the fields based on your specific keys.
        try:
            parsed = {}
            
            # A. Extract Text Fields
            # Captures text between: "key": " ...AND... ", (comma) or "} (end brace)
            str_keys = [
                "holistic_redescription", 
                "social_actors_analysis", 
                "populist_explanation", 
                "elitist_explanation", 
                "intensity_explanation"
            ]
            
            for key in str_keys:
                match = re.search(f'"{key}"\s*:\s*"(.*?)"\s*(?:,|}}\s*$)', answer, re.DOTALL)
                if match:
                    # Clean up: Replace internal double quotes with single quotes to be safe
                    parsed[key] = match.group(1).replace('"', "'")
                else:
                    parsed[key] = None

            # B. Extract Number Fields
            num_keys = ["populist_score", "elitist_score", "intensity_score"]
            for key in num_keys:
                match = re.search(f'"{key}"\s*:\s*(-?\d+)', answer)
                if match:
                    parsed[key] = int(match.group(1))
                else:
                    parsed[key] = None
            
            answer_data = parsed
            
        except Exception as regex_error:
            # If both fail, log it
            answer_data = {"error": f"Parse Failed: {regex_error}", "raw_output": answer}

    # --- 3. Append to Results ---
    d_results.append({
        "id": row.id, 
        "xid": row.xid, 
        "text": row.text,
        **answer_data
    })

# Convert to DataFrame
d = pd.DataFrame(d_results)

AttributeError: 'str' object has no attribute 'outputs'

In [11]:
d_results = []
for row, output in zip(rows_list, outputs):
    answer = output  # <-- changed from output.outputs[0].text

    # --- 1. Attempt Standard JSON Parsing ---
    try:
        json_str = answer[answer.find("{"):answer.rfind("}") + 1]
        answer_data = json.loads(json_str)
    except (json.JSONDecodeError, ValueError, AttributeError):
        # --- 2. Regex Fallback ---
        try:
            parsed = {}
            str_keys = [
                "holistic_redescription", 
                "social_actors_analysis", 
                "populist_explanation", 
                "elitist_explanation", 
                "intensity_explanation"
            ]
            for key in str_keys:
                match = re.search(f'"{key}"\s*:\s*"(.*?)"\s*(?:,|}}\s*$)', answer, re.DOTALL)
                if match:
                    parsed[key] = match.group(1).replace('"', "'")
                else:
                    parsed[key] = None

            num_keys = ["populist_score", "elitist_score", "intensity_score"]
            for key in num_keys:
                match = re.search(f'"{key}"\s*:\s*(-?\d+)', answer)
                if match:
                    parsed[key] = int(match.group(1))
                else:
                    parsed[key] = None

            answer_data = parsed
        except Exception as regex_error:
            answer_data = {"error": f"Parse Failed: {regex_error}", "raw_output": answer}

    # --- 3. Append to Results ---
    d_results.append({
        "id": row.id, 
        "text": row.text,
        **answer_data
    })

d = pd.DataFrame(d_results)

# Save Results

In [7]:
d.to_parquet(f'{data_path}/llm_coding.parquet')